# LOAD DATA FILE GENERATED BY STEP

## Import Libraries

In [38]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore")
warnings.filterwarnings( "ignore", module = "pandas\..*" )

In [39]:
import os
import datetime
import importlib
import utils as ut
import numpy as np
import pandas as pd

In [40]:
exp_type = 'E6'

## Path Setting

In [41]:
submission_version = False

if submission_version:
    DS_path_main = '\notebooks\dataset\mvtec_anomaly_detection'
else:
    DS_path_main = 'D:\DS\AD\mvtec_preprocessed'

DS_name = 'hazelnut'
DS_path = os.path.join(DS_path_main,DS_name)
codes_path = os.getcwd()
results_path=os.path.join(codes_path,'results')
results_subs        = os.path.join(results_path,    'test_results')
path_csv    = os.path.join(results_path,     'csv')

# Default Parameters

In [42]:
num_epochs = 30000
num_explained_classes = 1
batch_size = 16
num_samples = 500
postfix_csv = f'{DS_name}_9'

exp_name = 'exp_csv_E6'

exp_no = 'E6'

In [43]:
def get_file_modify_time(filepath=None):
    if os.path.exists(filepath):
        datestamp  = datetime.datetime.fromtimestamp(os.path.getmtime(filepath))
        print('File Exists :', filepath)
        print('File Date/Time :', datestamp)

# Read Pre-Computed CSV 

In [44]:
csv_filename = f'{path_csv}/{exp_name}_testresults_{postfix_csv}_BPT_new_eval.csv'
print(csv_filename,' *** ',os.path.exists(csv_filename))

e:\Cloud\RashidPHD\Publication_Data\QualITA\notebooks\results\csv/exp_csv_E6_testresults_hazelnut_9_BPT_new_eval.csv  ***  True


In [45]:
df = pd.read_csv(csv_filename)
df.head(3)

,Unnamed: 0,a_type_id,a_type,img_no,mask_type,target_segs,num_samples,anomaly_score,anomaly_score_val,AM_roc_score,...,aucD_clipr,threshold,best_point,max_IoU,au_IoU,time_exp,time_aucI,time_aucD,time_IoU,time_total
0,0,0,crack,0,blend_2,100,500,0.552117,173.42245,0.949351,...,0.096924,0.069475,0.078491,0.681358,0.195403,2.082480,0.244972,0.236366,0.062511,3.075715
1,1,0,crack,0,blend_2,100,500,0.552117,173.42245,0.949351,...,0.087667,0.046827,0.068665,0.779786,0.194305,9.265343,0.466010,0.615621,0.084360,10.880720
2,2,0,crack,0,blend_2,100,500,0.552117,173.42245,0.949351,...,0.086206,0.030308,0.076538,0.711409,0.192212,14.514923,0.435152,0.529823,0.057493,15.986776


In [46]:
print('Total Time: \t\t',str(datetime.timedelta(seconds=np.sum(df.time_total))))

Total Time: 		 1:37:50.147162


In [47]:
importlib.reload(ut)   
train_dir,gtruth_dir,test_dir = ut.Data.get_subdata(DS_path=DS_path)
anomaly_types    = sorted(os.listdir(test_dir))
anomaly_types_gt = sorted(os.listdir(gtruth_dir))
anomaly_types,anomaly_types_gt = ut.Data.get_anomaly_types(test_dir=test_dir,dataset=DS_name,gt_dir=gtruth_dir)
image_counter_cls = []
for i in range(len(anomaly_types)):
    image_counter_cls.extend([[anomaly_types[i],len(os.listdir(os.path.join(test_dir,anomaly_types[i])))]])
train_dir,gtruth_dir,test_dir = ut.Data.get_subdata(DS_path=DS_path)

augmentation_target = 'custom' # 'medium', 'full' minimal , custom
train_generator,test_generator, gtruth_generator= ut.Data2.load_data(train_dir = train_dir,
                                                                     test_dir = test_dir,
                                                                     gtruth_dir = gtruth_dir,
                                                                     dataset=DS_name,
                                                                     classes = anomaly_types,
                                                                     augmentation_target=augmentation_target,
                                                                    )

************************************************************************************************************************
Classes in Test:	['crack', 'cut', 'good', 'hole', 'print']
Classes in GT:		['crack', 'cut', 'hole', 'print']
************************************************************************************************************************
Classes in Test:	['good', 'crack', 'cut', 'hole', 'print']
Found 391 images belonging to 1 classes.
Found 110 images belonging to 5 classes.
Found 70 images belonging to 4 classes.


In [48]:
image_counter = {class_name: len(os.listdir(os.path.join(test_dir, class_name))) for class_name in test_generator.class_indices}
print(image_counter)

min_image_count = min(image_counter.items(), key=lambda x: x[1])[1]
min_image_class = min(image_counter.items(), key=lambda x: x[1])[0]
print(min_image_class,min_image_count)
from collections import Counter
Counter(test_generator.classes)
lbls_cls = list(image_counter.keys())
count_cls = list(image_counter.values())
lbls_cls,count_cls

{'good': 40, 'crack': 18, 'cut': 17, 'hole': 18, 'print': 17}
cut 17


(['good', 'crack', 'cut', 'hole', 'print'], [40, 18, 17, 18, 17])

# Read Pre-Computed CSV File for Anomaly Scores

In [49]:
postfix_csv = f'{DS_name}_{num_epochs}_AS'
ascsv_filename = f'{path_csv}/testresults_{postfix_csv}.csv'
df_as= pd.read_csv(ascsv_filename)

In [50]:
anomaly_scores = list(df_as[['anomaly_score_new', 'good_or_bad', 'a_type_id', 'img_no', 'a_type']].itertuples(index=False, name=None))

## FIND Delta Opt

In [51]:
delta_opt = ut.Evaluate.find_optimal_separation_threshold(anomaly_scores)
print(f'delta_opt\t:\t{delta_opt}')

36 (0.36697295, False, 0, 36, 'good')
delta_opt	:	0.36698295000000003


## GENERATE PDF PAGES

In [52]:
mask_type = df.mask_type.unique()[0]

In [83]:
import os
import json
import numpy as np
from datetime import datetime


# ── Available themes ──────────────────────────────────────────────────────────
#
#   'cyber'    – dark navy + cyan + hot-pink   (default)
#   'dracula'  – dark purple + violet + green
#   'amber'    – deep black + amber + orange
#   'light'    – clean white + indigo + coral
#   'forest'   – dark green + lime + teal
#
# Pass  theme=<name>  to generate_html() to bake in a default.
# The output HTML always includes a live theme-switcher bar so the viewer
# can switch themes in the browser — no regeneration needed.
# ─────────────────────────────────────────────────────────────────────────────

THEMES = {
    "cyber": {
        "bg":        "#080c10",
        "surface":   "#0d1117",
        "surface2":  "#161b22",
        "border":    "#21262d",
        "accent":    "#00e5ff",
        "accent2":   "#ff3c6e",
        "good":      "#ff3c6e",
        "bad":       "#23d18b",
        "text":      "#e6edf3",
        "muted":     "#7d8590",
        "hero_bg":   "linear-gradient(135deg, #0d1117 0%, #0a0f14 100%)",
        "glow_rgb":  "0,229,255",
        "scanlines": True,
    },
    "dracula": {
        "bg":        "#191823",
        "surface":   "#21202e",
        "surface2":  "#2a2740",
        "border":    "#3d3a55",
        "accent":    "#bd93f9",
        "accent2":   "#50fa7b",
        "good":      "#50fa7b",
        "bad":       "#ff5555",
        "text":      "#f8f8f2",
        "muted":     "#6272a4",
        "hero_bg":   "linear-gradient(135deg, #21202e 0%, #191823 100%)",
        "glow_rgb":  "189,147,249",
        "scanlines": False,
    },
    "amber": {
        "bg":        "#0a0800",
        "surface":   "#120f00",
        "surface2":  "#1c1800",
        "border":    "#2e2600",
        "accent":    "#ffb300",
        "accent2":   "#ff6d00",
        "good":      "#69f0ae",
        "bad":       "#ff6d00",
        "text":      "#fff8e1",
        "muted":     "#8d7b40",
        "hero_bg":   "linear-gradient(135deg, #120f00 0%, #0a0800 100%)",
        "glow_rgb":  "255,179,0",
        "scanlines": True,
    },
    "light": {
        "bg":        "#f5f7fa",
        "surface":   "#ffffff",
        "surface2":  "#eef1f6",
        "border":    "#d0d7e2",
        "accent":    "#4361ee",
        "accent2":   "#f72585",
        "good":      "#06d6a0",
        "bad":       "#ef233c",
        "text":      "#1a1d2e",
        "muted":     "#6b7280",
        "hero_bg":   "linear-gradient(135deg, #eef1f6 0%, #e8ecf4 100%)",
        "glow_rgb":  "67,97,238",
        "scanlines": False,
    },
    "forest": {
        "bg":        "#050f08",
        "surface":   "#0a1a0f",
        "surface2":  "#112218",
        "border":    "#1d3728",
        "accent":    "#39ff14",
        "accent2":   "#00e8c6",
        "good":      "#39ff14",
        "bad":       "#ff4747",
        "text":      "#d4f5dc",
        "muted":     "#5a8a65",
        "hero_bg":   "linear-gradient(135deg, #0a1a0f 0%, #050f08 100%)",
        "glow_rgb":  "57,255,20",
        "scanlines": False,
    },
}

VALID_THEMES = list(THEMES.keys())


def _build_css(t: dict) -> str:
    scanline_rule = """
      body::before {
        content: '';
        position: fixed; inset: 0;
        pointer-events: none;
        z-index: 9999;
      }""" if t["scanlines"] else ""

    return f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=Syne:wght@400;600;800&display=swap');

      :root {{
        --bg:       {t['bg']};
        --surface:  {t['surface']};
        --surface2: {t['surface2']};
        --border:   {t['border']};
        --accent:   {t['accent']};
        --accent2:  {t['accent2']};
        --good:     {t['good']};
        --bad:      {t['bad']};
        --text:     {t['text']};
        --muted:    {t['muted']};
        --glow:     rgba({t['glow_rgb']}, 0.14);
      }}

      *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
      html {{ scroll-behavior: smooth; }}

      body {{
        background: var(--bg);
        color: var(--text);
        font-family: 'Space Mono', monospace;
        min-height: 100vh;
        overflow-x: hidden;
      }}
      {scanline_rule}

      /* ─── Theme switcher bar ─── */
      .theme-bar {{
        display: flex;
        align-items: center;
        gap: 8px;
        padding: 10px 24px;
        background: var(--surface2);
        border-bottom: 1px solid var(--border);
        font-size: 10px;
        letter-spacing: 0.18em;
        text-transform: uppercase;
        color: var(--muted);
        flex-wrap: wrap;
      }}
      .theme-bar > span {{ margin-right: 4px; flex-shrink: 0; }}
      .theme-btn {{
        padding: 4px 14px;
        border: 1px solid var(--border);
        border-radius: 20px;
        background: transparent;
        color: var(--muted);
        font-family: 'Space Mono', monospace;
        font-size: 10px;
        letter-spacing: 0.12em;
        cursor: pointer;
        transition: all 0.18s ease;
      }}
      .theme-btn:hover {{ border-color: var(--accent); color: var(--accent); }}
      .theme-btn.active {{
        background: var(--accent);
        color: var(--bg);
        border-color: var(--accent);
        font-weight: 700;
      }}

      /* ─── Hero ─── */
      .hero {{
        position: relative;
        padding: 64px 40px 48px;
        border-bottom: 1px solid var(--border);
        overflow: hidden;
        background: {t['hero_bg']};
      }}
      .hero::after {{
        content: '';
        position: absolute;
        top: -120px; right: -120px;
        width: 480px; height: 480px;
        border-radius: 50%;
        pointer-events: none;
      }}
      .hero-eyebrow {{
        font-size: 11px; letter-spacing: 0.3em;
        text-transform: uppercase;
        color: var(--accent); margin-bottom: 18px; opacity: 0.85;
      }}
      .hero h1 {{
        font-family: 'Syne', sans-serif;
        font-weight: 500;
        font-size: clamp(18px, 4vw, 52px);
        line-height: 1.1; color: var(--text);
      }}
      .hero h1 em {{ font-style: normal; color: var(--accent); }}
      .hero-meta {{ margin-top: 18px; display: flex; flex-wrap: wrap; gap: 12px; }}
      .tag {{
        display: inline-flex; align-items: center; gap: 6px;
        padding: 5px 12px;
        border: 1px solid var(--border); border-radius: 4px;
        font-size: 11px; letter-spacing: 0.08em;
        color: var(--muted); background: var(--surface2);
      }}
      .tag span {{ color: var(--text); }}

      /* ─── Stats bar ─── */
      .stats-bar {{
        display: flex;
        border-bottom: 1px solid var(--border);
        background: var(--surface);
      }}
      .stat {{
        flex: 1; padding: 20px 24px;
        border-right: 1px solid var(--border);
        position: relative;
      }}
      .stat:last-child {{ border-right: none; }}
      .stat-value {{
        font-family: 'Syne', sans-serif;
        font-size: 26px; font-weight:500;
        color: var(--accent); line-height: 1; margin-bottom: 4px;
      }}
      .stat-label {{
        font-size: 10px; letter-spacing: 0.2em;
        text-transform: uppercase; color: var(--muted);
      }}
      .stat::before {{
        content: ''; position: absolute;
        top: 0; left: 0; width: 2px; height: 100%;
        background: var(--accent); opacity: 0.45;
      }}

      /* ─── Content ─── */
      .content {{ padding: 32px 24px; max-width: 1800px; margin: 0 auto; }}

      .section-head {{
        display: flex; align-items: center; gap: 12px;
        margin: 40px 0 20px; padding-bottom: 12px;
        border-bottom: 1px solid var(--border);
      }}
      .section-head-dot {{
        width: 8px; height: 8px; border-radius: 50%;
        background: var(--accent2);
        box-shadow: 0 0 10px var(--accent2);
      }}
      .section-head-title {{
        font-family: 'Syne', sans-serif;
        font-weight: 600; font-size: 13px;
        letter-spacing: 0.25em; text-transform: uppercase; color: var(--text);
      }}
      .section-head-count {{ margin-left: auto; font-size: 11px; color: var(--muted); }}

      /* ─── Table ─── */
      .table-wrap {{
        overflow-x: auto;
        border: 1px solid var(--border);
        border-radius: 8px;
        background: var(--surface);
      }}
      table {{ width: 100%; border-collapse: collapse; table-layout: fixed; }}
      thead tr {{ background: var(--surface2); border-bottom: 2px solid var(--border); }}
      th {{
        padding: 14px 10px;
        font-size: 10px; font-weight: 700;
        letter-spacing: 0.2em; text-transform: uppercase;
        color: var(--muted); text-align: center;
        border-right: 1px solid var(--border);
        position: sticky; top: 0; z-index: 10;
        background: var(--surface2); white-space: nowrap;
      }}
      th:last-child {{ border-right: none; }}
      tr.img-row td {{ padding: 0; border-bottom: 1px solid var(--border); }}
      tr.img-row td img {{
        display: block; width: 100%; height: auto; transition: filter 0.2s;
      }}
      tr.img-row td img:hover {{ filter: brightness(1.07) contrast(1.04); }}
      tr.caption-row td {{
        padding: 10px 16px;
        border-bottom: 2px solid var(--border);
        background: var(--surface2);
      }}
      .caption-inner {{
        display: flex; align-items: center; gap: 10px; font-size: 12px;
      }}
      .dot-indicator {{ width: 10px; height: 10px; border-radius: 50%; flex-shrink: 0; }}
      .dot-good {{ background: var(--good);  box-shadow: 0 0 8px var(--good); }}
      .dot-bad  {{ background: var(--bad);   box-shadow: 0 0 8px var(--bad); }}
      .caption-id    {{ color: var(--muted); }}
      .caption-score {{ font-weight: 700; color: var(--text); letter-spacing: 0.05em; }}
      .caption-badge {{
        padding: 2px 8px; border-radius: 3px;
        font-size: 10px; letter-spacing: 0.15em;
        text-transform: uppercase; font-weight: 700;
      }}
      .badge-anom {{ background: rgba(35,209,139,0.12); color: var(--bad);  border: 1px solid rgba(255,60,110,0.3); }}
      .badge-good {{ background: rgba(255,60,110,0.15); color: var(--good); border: 1px solid rgba(35,209,139,0.25); }}

      /* ─── Footer ─── */
      footer {{
        margin-top: 60px; padding: 24px 40px;
        border-top: 1px solid var(--border);
        display: flex; justify-content: space-between; align-items: center;
        font-size: 11px; color: var(--muted); letter-spacing: 0.08em;
      }}

      /* ─── Scrollbar ─── */
      ::-webkit-scrollbar {{ width: 6px; height: 6px; }}
      ::-webkit-scrollbar-track {{ background: var(--bg); }}
      ::-webkit-scrollbar-thumb {{ background: var(--border); border-radius: 3px; }}
      ::-webkit-scrollbar-thumb:hover {{ background: var(--muted); }}
    </style>"""


def _build_theme_js(active: str) -> str:
    """Inline JS for the runtime theme-switcher (persists via localStorage)."""
    return f"""
    <script>
    const THEMES = {json.dumps(THEMES)};

    function applyTheme(name) {{
      const t = THEMES[name];
      if (!t) return;
      const s = document.documentElement.style;
      s.setProperty('--bg',       t.bg);
      s.setProperty('--surface',  t.surface);
      s.setProperty('--surface2', t.surface2);
      s.setProperty('--border',   t.border);
      s.setProperty('--accent',   t.accent);
      s.setProperty('--accent2',  t.accent2);
      s.setProperty('--good',     t.good);
      s.setProperty('--bad',      t.bad);
      s.setProperty('--text',     t.text);
      s.setProperty('--muted',    t.muted);
      s.setProperty('--glow',     'rgba(' + t.glow_rgb + ', 0.14)');
      const hero = document.querySelector('.hero');
      if (hero) hero.style.background = t.hero_bg;
      document.querySelectorAll('.theme-btn').forEach(btn => {{
        btn.classList.toggle('active', btn.dataset.theme === name);
      }});
      try {{ localStorage.setItem('shapbpt_theme', name); }} catch(e) {{}}
    }}

    window.addEventListener('DOMContentLoaded', () => {{
      let saved = '{active}';
      try {{ saved = localStorage.getItem('shapbpt_theme') || '{active}'; }} catch(e) {{}}
      applyTheme(saved);
    }});
    </script>"""


def generate_html(
    img_types,
    results_path,
    exp_type,
    DS_name,
    anomaly_types,
    df_as,
    df,
    delta_opt,
    epochs,
    mask_type,
    image_counter,
    results_subs,
    theme: str = "cyber",
):
    """
    Generate a styled HTML results report for ShapBPT.

    Parameters
    ----------
    theme : str
        Default color theme baked into the file.
        The HTML also ships a live theme-switcher so viewers can change it
        in their browser without regenerating.

        Available themes
        ----------------
        'cyber'   – dark navy  + cyan  + hot-pink  (default)
        'dracula' – dark purple + violet + green
        'amber'   – deep black + amber + orange
        'light'   – clean white + indigo + coral
        'forest'  – dark green + lime  + teal
    """
    if isinstance(img_types, str):
        img_types = [img_types]

    if theme not in THEMES:
        raise ValueError(
            f"Unknown theme '{theme}'. Valid options: {', '.join(VALID_THEMES)}"
        )

    html_f_name   = f'{results_path}/HTML_{exp_type}_{DS_name}_{"_".join(img_types)}.html'
    total_images  = sum(image_counter.get(a, 0) for a in anomaly_types if a != 'good')
    anomaly_count = len([a for a in anomaly_types if a != 'good'])
    method_names  = ['Input', 'Recons', 'Anom Map'] + list(df.method.unique()) + ['Gt']
    generated_at  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    t   = THEMES[theme]
    css = _build_css(t)
    js  = _build_theme_js(theme)

    theme_buttons = " ".join(
        f'<button class="theme-btn{"  active" if name == theme else ""}" '
        f'data-theme="{name}" onclick="applyTheme(\'{name}\')">{name}</button>'
        for name in VALID_THEMES
    )

    with open(html_f_name, 'w') as f:

        f.write(f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>ShapBPT · {DS_name} · Results</title>
  {css}
</head>
<body>

<!-- ═══ THEME BAR ═══ -->
<div class="theme-bar">
  <span>Theme</span>
  {theme_buttons}
</div>

<!-- ═══ HERO ═══ -->
<div class="hero">
  <div class="hero-eyebrow">ShapBPT · Image Feature Attributions</div>
  <h1>Explainable Anomaly Detection using <em>Shap-BPT</em></h1>
  <div class="hero-meta">
    <div class="tag">DATASET <span>{DS_name}</span></div>
    <div class="tag">EXPERIMENT <span>{exp_type}</span></div>
    <div class="tag">EPOCHS <span>{epochs}</span></div>
    <div class="tag">MASK <span>{mask_type}</span></div>
    <div class="tag">δ_opt <span>{delta_opt:.4f}</span></div>
    <div class="tag">GENERATED <span>{generated_at}</span></div>
  </div>
</div>

<!-- ═══ STATS BAR ═══ -->
<div class="stats-bar">
  <div class="stat">
    <div class="stat-value">{total_images}</div>
    <div class="stat-label">Total Samples</div>
  </div>
  <div class="stat">
    <div class="stat-value">{anomaly_count}</div>
    <div class="stat-label">Anomaly Classes</div>
  </div>
  <div class="stat">
    <div class="stat-value">{len(method_names)}</div>
    <div class="stat-label">Methods</div>
  </div>
  <div class="stat">
    <div class="stat-value">{len(img_types)}</div>
    <div class="stat-label">Image Types</div>
  </div>
</div>

<!-- ═══ CONTENT ═══ -->
<div class="content">
""")

        for a_type in anomaly_types:
            if a_type == 'good':
                continue
            count = image_counter.get(a_type, 0)
            if count == 0:
                continue

            f.write(f"""
  <div class="section-head">
    <div class="section-head-dot"></div>
    <div class="section-head-title">{a_type}</div>
    <div class="section-head-count">{count} samples</div>
  </div>

  <div class="table-wrap">
  <table>
    <thead><tr>
""")
            for method_name in method_names:
                f.write(f"      <th>{method_name}</th>\n")
            f.write("    </tr></thead>\n    <tbody>\n")

            for img_id in range(count):
                results_subpath_cls = os.path.join(results_subs, str(a_type))

                asn      = df_as[(df_as.a_type == a_type) & (df_as.img_no == img_id)].anomaly_score_new
                asn_bool = asn > delta_opt
                is_anom  = asn_bool.iloc[0]
                pred_label = 'Anomalous' if is_anom else 'Normal'
                dot_cls    = 'dot-bad'    if is_anom else 'dot-good'
                badge_cls  = 'badge-anom' if is_anom else 'badge-good'

                valid_images = []
                for img_type in img_types:
                    p = os.path.abspath(
                        f'{results_subpath_cls}/{img_id}_{img_type}_{a_type}_{img_id}_{epochs}_{mask_type}.png'
                    )
                    if os.path.exists(p):
                        valid_images.append(p)

                if not valid_images:
                    continue

                for img_path in valid_images:
                    f.write(f"""      <tr class="img-row">
        <td colspan="{len(method_names)}"><img src="{img_path}" loading="lazy" alt="{a_type} #{img_id}"></td>
      </tr>
""")
                f.write(f"""      <tr class="caption-row">
        <td colspan="{len(method_names)}">
          <div class="caption-inner">
            <span class="dot-indicator {dot_cls}"></span>
            <span class="caption-id">{a_type} · #{img_id:03d}</span>
            <span class="caption-score">score: {asn.iloc[0]:.4f}</span>
            <span class="caption-badge {badge_cls}">{pred_label}</span>
          </div>
        </td>
      </tr>
""")

            f.write("    </tbody>\n  </table>\n  </div>\n")

        f.write(f"""
</div><!-- /.content -->

<footer>
  <div>ShapBPT · {DS_name} · {exp_type}</div>
  <div>Generated {generated_at}</div>
</footer>

{js}
</body>
</html>
""")

    print(f'✓  Report written → {html_f_name}  (default theme: {theme})')


In [84]:
# img_types = 'heatmaps'
# img_types = 'IoU'
img_types = ['heatmaps','IoU']

generate_html(img_types, results_path,exp_type, DS_name, anomaly_types, df_as, df, delta_opt, num_epochs, mask_type, image_counter, results_subs)


✓  Report written → e:\Cloud\RashidPHD\Publication_Data\QualITA\notebooks\results/HTML_E6_hazelnut_heatmaps_IoU.html  (default theme: cyber)
